# Influential data identification - Llama2 - Svamp

This notebook demonstrates how to efficiently compute the influence functions using RRInf, showing its application to **influential data identification** tasks.

- Model: [llama-2-13b-chat](https://huggingface.co/meta-llama/Llama-2-13b-chat-hf) trained on a mix of publicly available online datasets.
- Fine-tuning dataset: [SVAMP](https://huggingface.co/datasets/ChilleD/SVAMP) Math Word Problems dataset.

References
- `trl` HuggingFace library [[Link]](https://github.com/huggingface/trl).

In [ ]:
import sys
sys.path.append('./src')
from lora_model import LORAEngineGeneration
from influence import IFEngineGeneration

import warnings
warnings.filterwarnings("ignore")

## Fine-tune a model
- We fine-tune a llama-2-13b-chat model on the `SVAMP` dataset. We use `src/sft_trainer.py`, which is built on HuggingFace's [SFTTrainer](https://github.com/huggingface/trl/blob/main/examples/scripts/sft.py). It will take around 30 minutes.
- `SVAMP` comprises 4 types. We choose 75 (resp., 25) examples for each type from the original train (resp., test) splits. The data can be found in `RRInf/datasets`.

In [ ]:
# !python /YOUR-RRINF-PATH/RRInf/src/sft_trainer.py \
#     --model_name /YOUR-LLAMA-PATH/llama/models_hf/llama-2-13b-chat \
#     --dataset_name /YOUR-RRINF-PATH/RRInf/datasets/svamp_train.hf \
#     --output_dir /YOUR-RRINF-PATH/RRInf/models/svamp_13bf \
#     --dataset_text_field text \
#     --load_in_4bit \
#     --use_peft

## Load a fine-tuned model

In [ ]:
# Please change the following objects to  "YOUR-LLAMA-PATH" and "YOUR-RRINF-PATH"
base_path = "/YOUR-LLAMA-PATH/llama/models_hf/llama-2-13b-chat"
project_path ="/YOUR-RRINF-PATH/RRInf"
lora_engine = LORAEngineGeneration(base_path=base_path,
                                   project_path=project_path,
                                   dataset_name='svamp')

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Some weights of LlamaForCausalLM were not initialized from the model checkpoint at ./models/Llama-2-13b-chat-hf and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Compute the gradient
 - Influence function uses the first-order gradient of a loss function. Here we compute gradients using `compute_gradient`
 - `tr_grad_dict` has a nested structure of two Python dictionaries. The outer dictionary has `{an index of the training data: a dictionary of gradients}` and the inner dictionary has `{layer name: gradients}`. The `val_grad_dict` has the same structure but for the validationd data points.

In [ ]:
import torch
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

In [ ]:
tokenized_datasets, collate_fn = lora_engine.create_tokenized_datasets()
tr_grad_dict, val_grad_dict = lora_engine.compute_gradient(tokenized_datasets, collate_fn)


Parameter 'function'=<function LORAEngineGeneration.create_tokenized_datasets.<locals>.<lambda> at 0x7e0903b628e0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

100%|██████████| 100/100 [00:58<00:00,  1.70it/s]


## Compute the influence function
 - We compute the baseline methods using `compute_IF_baselines()` including `Hessien-free` and `DataInf`.
 - We compute `RRInf` using `compute_IF_RRInf()` which randomly samples a neuron at every iteration by setting `layer=False`.

In [ ]:
influence_engine = IFEngineGeneration()
influence_engine.preprocess_gradients(tr_grad_dict, val_grad_dict)
influence_engine.compute_IF_baselines()
influence_engine.compute_IF_RRInf(num_iterations=1000,learning_rate=0.01,layer=False)

## Attributes of influence_engine
To compare the runtime, one can use `time_dict` attribute in `influence_engine`.

In [ ]:
influence_engine.time_dict

defaultdict(list,
            {'identity': 187.11856317520142,
             'DataInf': 1125.3178534507751,
             'RRInf': 0.5774827003479004})

In [ ]:
influence_engine.IF_dict.keys()

dict_keys(['identity', 'DataInf', 'RRInf'])

# AUC and Recall

In [ ]:
from datasets import load_dataset, load_from_disk
train_dataset = load_from_disk('./datasets/svamp_train.hf')
validation_dataset = load_from_disk('./datasets/svamp_test.hf')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

identity_df=influence_engine.IF_dict['identity']
datainf_df=influence_engine.IF_dict['DataInf']
rrinf_df=influence_engine.IF_dict['RRInf']

identity_auc_list, datainf_auc_list, rrinf_auc_list=[], [], []
for i in range(len(validation_dataset['variation'])):
    gt_label=validation_dataset['variation'][i]
    gt_array=np.array([1 if tr_label == gt_label else 0 for tr_label in train_dataset['variation']])


    # The influence function is anticipated to have a big negative value when its class equals to a validation data point.
    # This is because a data point with the same class is likely to be more helpful in minimizing the validation loss.
    # Thus, we multiply the influence function value by -1 to account for alignment with the gt_array.
    identity_auc_list.append(roc_auc_score(gt_array, -(identity_df.iloc[i,:].to_numpy())))
    datainf_auc_list.append(roc_auc_score(gt_array, -(datainf_df.iloc[i,:].to_numpy())))
    rrinf_auc_list.append(roc_auc_score(gt_array, -(rrinf_df.iloc[i,:].to_numpy())))

print(f'identity AUC: {np.mean(identity_auc_list):.3f}/{np.std(identity_auc_list):.3f}')
print(f'DataInf AUC: {np.mean(datainf_auc_list):.3f}/{np.std(datainf_auc_list):.3f}')
print(f'RRInf AUC: {np.mean(rrinf_auc_list):.3f}/{np.std(rrinf_auc_list):.3f}')

identity AUC: 0.600/0.076
DataInf AUC: 0.657/0.091
RRInf AUC: 0.669/0.080


In [ ]:
# Recall calculations
train_array=np.array(train_dataset['variation'])
identity_recall_list, datainf_recall_list, rrinf_recall_list=[], [], []
for i in range(len(validation_dataset['variation'])):
    gt_label=validation_dataset['variation'][i]
    n_label=np.sum(train_array == gt_label)


    # Similar to AUC computation, we consider the first 90 data points with the smallest influence function values
    # These data points with the smallest influence function values likely have the same class with the validation data point.
    sorted_index=np.argsort(identity_df.iloc[i].values) # ascending order
    sorted_array=np.array([train_dataset['variation'][j] for j in sorted_index])
    recall_identity=np.count_nonzero(sorted_array[:n_label] == gt_label)/n_label
    identity_recall_list.append(recall_identity)

    sorted_index=np.argsort(datainf_df.iloc[i].values) # ascending order
    sorted_array=np.array([train_dataset['variation'][j] for j in sorted_index])
    recall_datainf=np.count_nonzero(sorted_array[:n_label] == gt_label)/n_label
    datainf_recall_list.append(recall_datainf)

    sorted_index = np.argsort(rrinf_df.iloc[i].values) # ascending order
    sorted_array=np.array([train_dataset['variation'][j] for j in sorted_index])
    recall_rrinf=np.count_nonzero(sorted_array[:n_label] == gt_label)/n_label
    rrinf_recall_list.append(recall_rrinf)

print(f'identity Recall: {np.mean(identity_recall_list):.3f}/{np.std(identity_recall_list):.3f}')
print(f'DataInf Recall: {np.mean(datainf_recall_list):.3f}/{np.std(datainf_recall_list):.3f}')
print(f'RRInf Recall: {np.mean(rrinf_recall_list):.3f}/{np.std(rrinf_recall_list):.3f}')

identity Recall: 0.330/0.083
DataInf Recall: 0.390/0.099
RRInf Recall: 0.419/0.095
